# BCMID - Data & Quality Agent
Ce notebook permet de regénérer tout le pipeline de prétraitement (images, rapports, anonymisation, splits) directement sur Kaggle.


### 1. Préparation de l'environnement
Création du dossier pour le module Python.


In [ ]:
!mkdir -p dq_agent
!mkdir -p tests


In [ ]:
%%writefile dq_agent/anonymize.py
"""Module 5 — Anonymization: SHA-256 pseudonymisation with environment salt."""

from __future__ import annotations

import csv
import hashlib
import logging
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger(__name__)

_SALT_ENV_VAR = "DQ_SALT"
_PSEUDO_PREFIX = "P"
_HASH_CHARS = 10


def _get_salt() -> str:
    """Read salt from environment variable or raise."""
    salt = os.environ.get(_SALT_ENV_VAR, "")
    if not salt:
        # Use a deterministic but non-empty default for development only
        logger.warning(
            "Environment variable %s not set — using default development salt. "
            "Set %s in your .env file for production!",
            _SALT_ENV_VAR,
            _SALT_ENV_VAR,
        )
        salt = "bcmid_dev_salt_change_me"
    return salt


def pseudonymise(patient_id: str, salt: str | None = None) -> str:
    """
    Generate a pseudonym for a patient ID.

    Format: P + first 10 hex chars of SHA-256(salt + patient_id)
    Deterministic: same inputs always produce the same output.
    """
    if salt is None:
        salt = _get_salt()
    h = hashlib.sha256((salt + patient_id).encode("utf-8")).hexdigest()
    return _PSEUDO_PREFIX + h[:_HASH_CHARS]


def build_mapping(patient_ids: list[str], salt: str | None = None) -> dict[str, str]:
    """
    Build a dict mapping real_id -> pseudo_id for all patients.

    Parameters
    ----------
    patient_ids : list of real patient folder names
    salt : override salt (for testing)
    """
    if salt is None:
        salt = _get_salt()

    mapping: dict[str, str] = {}
    seen_pseudos: set[str] = set()

    for pid in patient_ids:
        pseudo = pseudonymise(pid, salt)
        if pseudo in seen_pseudos:
            # Collision (astronomically rare) — extend hash
            h = hashlib.sha256((salt + pid + "extra").encode()).hexdigest()
            pseudo = _PSEUDO_PREFIX + h[:_HASH_CHARS]
        seen_pseudos.add(pseudo)
        mapping[pid] = pseudo

    return mapping


def save_private_mapping(mapping: dict[str, str], out_path: Path) -> None:
    """
    Save the real_id -> pseudo_id mapping to a PRIVATE CSV file.

    This file must NEVER be shared or committed to git.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["real_id", "pseudo_id"])
        for real_id, pseudo_id in sorted(mapping.items()):
            writer.writerow([real_id, pseudo_id])
    logger.info("Private mapping saved to %s (DO NOT SHARE)", out_path)


def load_private_mapping(path: Path) -> dict[str, str]:
    """Load real_id -> pseudo_id mapping from CSV."""
    mapping: dict[str, str] = {}
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            mapping[row["real_id"]] = row["pseudo_id"]
    return mapping



In [ ]:
%%writefile dq_agent/cli.py
"""
CLI entry point for the Data & Quality Agent.

Usage
-----
    python -m dq_agent run    --config config.yaml [--limit N]
    python -m dq_agent profile --config config.yaml [--limit N]
    python -m dq_agent splits  --config config.yaml
    python -m dq_agent report  --config config.yaml
"""

from __future__ import annotations

import argparse
import csv
import json
import logging
import sys
import zipfile
from pathlib import Path
from typing import Any

from .config import Config, load_config, setup_logging
from .splits import DEFAULT_LABEL_GROUPS, get_label_group

logger = logging.getLogger(__name__)


# ──────────────────────────── helpers ────────────────────────────────────────

def _load_progress(cfg: Config) -> dict[str, Any]:
    if cfg.progress_file.exists():
        with open(cfg.progress_file, encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_progress(cfg: Config, progress: dict[str, Any]) -> None:
    cfg.progress_file.parent.mkdir(parents=True, exist_ok=True)
    with open(cfg.progress_file, "w", encoding="utf-8") as f:
        json.dump(progress, f, indent=2)


def _write_manifests(
    cfg: Config,
    processed: list[dict],
    modalities: dict[str, dict],
    labels: dict[str, str],
    extra_data: dict[str, dict],
    mapping: dict[str, str],
    label_groups: dict[str, str],
) -> None:
    """Write manifest.json and manifest.csv — no real IDs anywhere."""
    out_dir = cfg.validated_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    reverse_map = {v: k for k, v in mapping.items()}
    records = []

    for item in processed:
        pseudo = item["pseudo_id"]
        real_id = reverse_map.get(pseudo, "")
        mod_info = modalities.get(real_id, {})
        birads = labels.get(real_id, "UNKNOWN")
        lbl_grp = get_label_group(birads, label_groups)
        ex = extra_data.get(real_id, {})

        records.append({
            "pseudo_id": pseudo,
            "n_us": item["n_us"],
            "n_mammo": item["n_mammo"],
            "us_paths": item["us_paths"],
            "mammo_paths": item["mammo_paths"],
            "modality_status": mod_info.get("status", "unknown"),
            "has_report": mod_info.get("has_report", False),
            "birads": birads,
            "label_group": lbl_grp,
            # Extra TSV columns (no real ID)
            "binary_label": ex.get("Binary Label", ""),
            "pathology_label": ex.get("Pathology Label", ""),
        })

    # JSON
    with open(out_dir / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    # CSV
    fieldnames = [
        "pseudo_id", "n_us", "n_mammo", "us_paths", "mammo_paths",
        "modality_status", "has_report", "birads", "label_group",
        "binary_label", "pathology_label",
    ]
    with open(out_dir / "manifest.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in records:
            row = dict(r)
            row["us_paths"] = ";".join(r["us_paths"])
            row["mammo_paths"] = ";".join(r["mammo_paths"])
            writer.writerow(row)

    logger.info("Manifests written → %s", out_dir)


# ──────────────────────────── commands ───────────────────────────────────────

def cmd_run(cfg: Config, limit: int | None) -> None:
    """Full pipeline."""
    from tqdm import tqdm

    from .anonymize import build_mapping, save_private_mapping
    from .integrity import check_integrity
    from .modality import classify_patients, filter_for_training, modality_summary
    from .plots import generate_all_plots
    from .preprocess import process_patient
    from .profiling import profile_dataset, save_profile
    from .report import build_report, save_report
    from .reports import process_reports
    from .splits import make_splits, save_splits
    from .validation import validate_labels
    from .zip_io import scan_zip

    # ── Directories
    for d in [cfg.work_dir, cfg.validated_dir, cfg.mammo_dir,
              cfg.us_dir, cfg.reports_dir, cfg.plots_dir]:
        d.mkdir(parents=True, exist_ok=True)

    progress = _load_progress(cfg)

    # ── Step 1: Scan ZIP
    if "patients" not in progress:
        logger.info("Scanning ZIP ...")
        all_patients_full = scan_zip(cfg.zip_path)
        n_total_zip = len(all_patients_full)
        patients = all_patients_full
        if limit:
            patients = dict(list(patients.items())[:limit])
            logger.info("--limit %d: keeping %d patients (total in ZIP: %d)", limit, len(patients), n_total_zip)
        progress["patients"] = patients
        progress["n_total_zip"] = n_total_zip
        _save_progress(cfg, progress)
    else:
        patients = progress["patients"]
        n_total_zip = progress.get("n_total_zip", len(patients))
        if limit:
            patients = dict(list(patients.items())[:limit])
        logger.info("Loaded %d patients from progress cache (total ZIP: %d)", len(patients), n_total_zip)

    # ── Step 2: Validate labels
    logger.info("Validating labels …")
    val_result = validate_labels(cfg, patients)
    labels = val_result["labels"]
    extra_data = val_result.get("extra_data", {})
    logger.info(
        "TSV columns: %s | ID col: '%s' | Label col: '%s' | Extra: %s",
        val_result["tsv_columns"], val_result["id_column"],
        val_result["label_column"], val_result.get("extra_columns", []),
    )

    # ── Step 3: Integrity
    logger.info("Checking image integrity …")
    integrity_result = check_integrity(cfg, patients)

    # ── Step 4: Modality classification (images only, not report)
    logger.info("Classifying modalities …")
    modalities = classify_patients(cfg, patients, integrity_result["valid_members"])
    multimodal_ids, partial_ids = filter_for_training(modalities)
    mod_counts = modality_summary(modalities)

    # ── Step 5: Profiling
    logger.info("Profiling dataset ...")
    profile = profile_dataset(cfg, patients, labels, n_total_zip_patients=n_total_zip)
    save_profile(profile, cfg.validated_dir)

    # ── Step 6: Anonymization
    logger.info("Building anonymization mapping …")
    all_pids = list(patients.keys())
    mapping = build_mapping(all_pids)
    save_private_mapping(mapping, cfg.mapping_file)

    # ── Step 7: Preprocess images (idempotent)
    logger.info("Preprocessing images …")
    processed_results: list[dict] = []

    with zipfile.ZipFile(cfg.zip_path, "r") as zf:
        for pid in tqdm(all_pids, desc="Preprocess", unit="patient"):
            pseudo = mapping[pid]
            mod_info = modalities[pid]

            first_us = cfg.us_dir / f"{pseudo}_us_0.png"
            first_mammo = cfg.mammo_dir / f"{pseudo}_mammo_0.png"
            has_us_done = (mod_info["n_valid_us"] == 0) or first_us.exists()
            has_mammo_done = (mod_info["n_valid_mammo"] == 0) or first_mammo.exists()

            if has_us_done and has_mammo_done:
                us_paths = sorted(str(p.relative_to(cfg.validated_dir))
                                  for p in cfg.us_dir.glob(f"{pseudo}_us_*.png"))
                mammo_paths = sorted(str(p.relative_to(cfg.validated_dir))
                                     for p in cfg.mammo_dir.glob(f"{pseudo}_mammo_*.png"))
                processed_results.append({
                    "pseudo_id": pseudo,
                    "us_paths": us_paths,
                    "mammo_paths": mammo_paths,
                    "n_us": len(us_paths),
                    "n_mammo": len(mammo_paths),
                })
                continue

            result = process_patient(pid, pseudo, mod_info, cfg, zf)
            processed_results.append(result)

    # ── Step 8: Extract PDF reports
    logger.info("Extracting PDF reports …")
    reports_stats = process_reports(
        patients=patients,
        mapping=mapping,
        zip_path=cfg.zip_path,
        reports_dir=cfg.reports_dir,
    )
    logger.info(
        "Label leakage risk: %.1f%% of reports mention BI-RADS",
        reports_stats["label_leakage_risk_pct"],
    )

    # ── Step 9: Write manifests (with label_group, no real IDs)
    logger.info("Writing manifests …")
    _write_manifests(cfg, processed_results, modalities, labels,
                     extra_data, mapping, DEFAULT_LABEL_GROUPS)

    # ── Step 10: Splits (stratified by label_group)
    pseudo_multimodal = [mapping[pid] for pid in multimodal_ids if pid in mapping]
    pseudo_labels = {mapping[pid]: labels.get(pid, "UNKNOWN")
                     for pid in all_pids if pid in mapping}
    splits = make_splits(cfg, pseudo_multimodal, pseudo_labels, DEFAULT_LABEL_GROUPS)
    save_splits(splits, cfg.validated_dir)

    # ── Step 11: Plots
    logger.info("Generating plots …")
    try:
        generate_all_plots(cfg, profile, mod_counts, patients, modalities)
    except Exception as exc:
        logger.warning("Plot generation failed (non-fatal): %s", exc)

    # ── Step 12: Quality report
    logger.info("Building quality report …")
    preprocessing_params = {
        "target_size": cfg.preprocessing.target_size,
        "mammo_clahe_clip_limit": cfg.preprocessing.mammo_clahe_clip_limit,
        "mammo_clahe_tile_grid": cfg.preprocessing.mammo_clahe_tile_grid,
        "us_median_ksize": cfg.preprocessing.us_median_ksize,
        "nlm_h": cfg.preprocessing.nlm_h,
        "bit_depth": cfg.preprocessing.bit_depth,
    }
    report = build_report(
        profile=profile,
        integrity_result=integrity_result,
        modality_counts=mod_counts,
        validation_result=val_result,
        preprocessing_params=preprocessing_params,
        n_multimodal_ready=len(multimodal_ids),
        reports_stats=reports_stats,
    )
    save_report(report, cfg.validated_dir)

    logger.info("✅ Pipeline complete! Outputs in: %s", cfg.validated_dir)
    _print_summary(report, splits, reports_stats)


def _print_summary(report: dict, splits: dict, reports_stats: dict) -> None:
    sep = "=" * 62
    print("\n" + sep)
    print("  DATA & QUALITY AGENT -- RESUME FINAL")
    print(sep)
    print(f"  Dossiers patients (ZIP)      : {report['n_patient_folders']}")
    print(f"  Patients annotes (TSV)       : {report['n_patients_in_annotations']}")
    print(f"  Ecart detecte                : {report['ecart_patients']}")
    print(f"  Prets (multimodal complet)   : {report['patients_ready_multimodal']}")
    print(f"  Sans rapport PDF             : {report['patients_without_report_pdf']}")
    print(f"  Images corrompues            : {report['corrupted_or_invalid_count']}")
    print(f"  Groupes de doublons          : {report['duplicate_files_count']}")
    print(f"  Sans label BI-RADS           : {len(report['patients_without_label'])}")
    print("-" * 62)
    print(f"  BI-RADS distribution         : {report['birads_distribution']}")
    print("-" * 62)
    print(f"  Rapports PDF extraits (OK)   : {reports_stats.get('reports_extracted_ok', 0)}")
    print(f"  Rapports scannes (illisibles): {reports_stats.get('reports_scanned_unreadable', 0)}")
    print(f"  Risque label leakage         : {reports_stats.get('label_leakage_risk_pct', 0):.1f}%")
    print("-" * 62)
    print(f"  Split train/val/test         : {splits['n_train']}/{splits['n_val']}/{splits['n_test']}")
    if report.get("labels_without_folder"):
        print(f"  [!] Labels sans dossier      : {len(report['labels_without_folder'])}")
    print(sep + "\n")



def cmd_profile(cfg: Config, limit: int | None) -> None:
    from .profiling import profile_dataset, save_profile
    from .validation import validate_labels
    from .zip_io import scan_zip

    patients = scan_zip(cfg.zip_path)
    if limit:
        patients = dict(list(patients.items())[:limit])

    val_result = validate_labels(cfg, patients)
    profile = profile_dataset(cfg, patients, val_result["labels"])
    save_profile(profile, cfg.validated_dir)

    print(f"\n{'='*50}")
    print(f"TSV columns     : {val_result['tsv_columns']}")
    print(f"ID column       : {val_result['id_column']}")
    print(f"Label column    : {val_result['label_column']}")
    print(f"Extra columns   : {val_result.get('extra_columns', [])}")
    print(f"Image extensions: {profile['image_extensions']}")
    print(f"Patients in ZIP : {len(patients)}")
    print(f"Patients in TSV : {val_result.get('n_patients_in_annotations', 'N/A')}")
    print(f"BI-RADS dist.   : {profile['birads_distribution']}")
    print(f"{'='*50}\n")


def cmd_splits(cfg: Config) -> None:
    from .splits import make_splits, save_splits, DEFAULT_LABEL_GROUPS

    manifest_path = cfg.validated_dir / "manifest.json"
    if not manifest_path.exists():
        print("ERROR: manifest.json not found — run `run` first.")
        sys.exit(1)

    with open(manifest_path, encoding="utf-8") as f:
        records = json.load(f)

    multimodal = [r["pseudo_id"] for r in records if r["modality_status"] == "complete"]
    labels = {r["pseudo_id"]: r["birads"] for r in records}

    splits = make_splits(cfg, multimodal, labels, DEFAULT_LABEL_GROUPS)
    save_splits(splits, cfg.validated_dir)
    print(f"Splits: train={splits['n_train']}, val={splits['n_val']}, test={splits['n_test']}")


def cmd_report(cfg: Config) -> None:
    print("Use `run` to regenerate the full report with all pipeline state.")


# ──────────────────────────── main ───────────────────────────────────────────

def main() -> None:
    parser = argparse.ArgumentParser(
        prog="python -m dq_agent",
        description="Data & Quality Agent — BCMID Multimodal Medical AI",
    )

    subparsers = parser.add_subparsers(dest="command", required=True)

    p_run = subparsers.add_parser("run", help="Run the full pipeline")
    p_run.add_argument("--config", default="config.yaml")
    p_run.add_argument("--limit", type=int, default=None)

    p_profile = subparsers.add_parser("profile", help="Profiling only")
    p_profile.add_argument("--config", default="config.yaml")
    p_profile.add_argument("--limit", type=int, default=None)

    p_splits = subparsers.add_parser("splits", help="Regenerate splits")
    p_splits.add_argument("--config", default="config.yaml")

    p_report = subparsers.add_parser("report", help="Regenerate report")
    p_report.add_argument("--config", default="config.yaml")

    args = parser.parse_args()
    cfg = load_config(args.config)
    setup_logging(cfg)

    if args.command == "run":
        cmd_run(cfg, args.limit)
    elif args.command == "profile":
        cmd_profile(cfg, args.limit)
    elif args.command == "splits":
        cmd_splits(cfg)
    elif args.command == "report":
        cmd_report(cfg)
    else:
        parser.print_help()
        sys.exit(1)


if __name__ == "__main__":
    main()



In [ ]:
%%writefile dq_agent/config.py
"""Configuration loader for the Data & Quality Agent."""

from __future__ import annotations

import os
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import yaml
from dotenv import load_dotenv

load_dotenv()  # charge .env si présent


@dataclass
class PreprocessingConfig:
    target_size: int = 512
    mammo_clahe_clip_limit: float = 2.0
    mammo_clahe_tile_grid: int = 8
    us_median_ksize: int = 3
    nlm_h: int = 5
    bit_depth: int = 8


@dataclass
class ModalityConfig:
    require_report: bool = False
    min_modalities: int = 2


@dataclass
class IntegrityConfig:
    min_dimension: int = 32
    std_threshold: float = 1.0


@dataclass
class SplitsConfig:
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    random_seed: int = 42
    rare_class_threshold: int = 5


@dataclass
class LoggingConfig:
    level: str = "INFO"
    file: str = "dq_agent.log"


@dataclass
class Config:
    zip_path: Path = Path("")
    tsv_path: Path = Path("")
    work_dir: Path = Path("bcmid_work")
    private_dir: Path = Path("bcmid_work")
    preprocessing: PreprocessingConfig = field(default_factory=PreprocessingConfig)
    modality_policy: ModalityConfig = field(default_factory=ModalityConfig)
    integrity: IntegrityConfig = field(default_factory=IntegrityConfig)
    splits: SplitsConfig = field(default_factory=SplitsConfig)
    logging: LoggingConfig = field(default_factory=LoggingConfig)

    @property
    def raw_dir(self) -> Path:
        return self.work_dir / "raw"

    @property
    def validated_dir(self) -> Path:
        return self.work_dir / "validated"

    @property
    def mammo_dir(self) -> Path:
        return self.validated_dir / "mammo"

    @property
    def us_dir(self) -> Path:
        return self.validated_dir / "us"

    @property
    def reports_dir(self) -> Path:
        return self.validated_dir / "reports"

    @property
    def plots_dir(self) -> Path:
        return self.validated_dir / "plots"

    @property
    def mapping_file(self) -> Path:
        """Private mapping — outside validated/."""
        return self.private_dir / "id_mapping_PRIVATE.csv"

    @property
    def progress_file(self) -> Path:
        return self.work_dir / ".progress.json"



def load_config(path: str | Path = "config.yaml") -> Config:
    """Load and validate configuration from a YAML file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Config file not found: {path}")

    with open(path, encoding="utf-8") as f:
        raw = yaml.safe_load(f)

    cfg = Config(
        zip_path=Path(raw.get("zip_path", "")),
        tsv_path=Path(raw.get("tsv_path", "")),
        work_dir=Path(raw.get("work_dir", "bcmid_work")),
        private_dir=Path(raw.get("private_dir", raw.get("work_dir", "bcmid_work"))),
    )

    if pp := raw.get("preprocessing"):
        cfg.preprocessing = PreprocessingConfig(**pp)

    if mp := raw.get("modality_policy"):
        cfg.modality_policy = ModalityConfig(**mp)

    if ig := raw.get("integrity"):
        cfg.integrity = IntegrityConfig(**ig)

    if sp := raw.get("splits"):
        cfg.splits = SplitsConfig(**sp)

    if lg := raw.get("logging"):
        cfg.logging = LoggingConfig(**lg)

    return cfg


def setup_logging(cfg: Config) -> None:
    """Configure root logger with file + console handlers."""
    level = getattr(logging, cfg.logging.level.upper(), logging.INFO)
    fmt = "%(asctime)s [%(levelname)s] %(name)s — %(message)s"
    handlers: list[logging.Handler] = [logging.StreamHandler()]
    log_path = cfg.work_dir / cfg.logging.file
    log_path.parent.mkdir(parents=True, exist_ok=True)
    handlers.append(logging.FileHandler(log_path, encoding="utf-8"))
    logging.basicConfig(level=level, format=fmt, handlers=handlers, force=True)



In [ ]:
%%writefile dq_agent/integrity.py
"""Module 2 — Integrity: corruption detection, duplicates, uniform images."""

from __future__ import annotations

import hashlib
import io
import logging
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image
from tqdm import tqdm

from .config import Config
from .zip_io import read_image_from_zip

logger = logging.getLogger(__name__)


def _hash_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _is_uniform(arr: np.ndarray, threshold: float) -> bool:
    """Return True if image standard deviation is below threshold (uninformative image)."""
    return float(arr.std()) < threshold


def check_integrity(
    cfg: Config,
    patients: dict[str, dict],
) -> dict[str, Any]:
    """
    Check integrity of all images in the ZIP.

    Returns
    -------
    dict with:
      - corrupted_or_invalid: list of {member, reason}
      - uniform_images: list of members
      - duplicate_files: list of lists (groups of identical files by hash)
      - valid_members: set of zip members that passed all checks
    """
    zip_path = cfg.zip_path
    min_dim = cfg.integrity.min_dimension
    std_thresh = cfg.integrity.std_threshold

    corrupted: list[dict] = []
    uniform: list[str] = []
    hash_map: dict[str, list[str]] = {}
    valid_members: set[str] = set()

    all_members: list[str] = []
    for inv in patients.values():
        all_members.extend(inv["us"] + inv["mammo"])

    logger.info("Checking integrity of %d images …", len(all_members))

    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in tqdm(all_members, desc="Integrity", unit="img"):
            # Read raw bytes for hashing
            try:
                data = zf.read(member)
            except Exception as exc:
                corrupted.append({"member": member, "reason": f"unreadable: {exc}"})
                continue

            # Hash for duplicate detection
            h = _hash_bytes(data)
            hash_map.setdefault(h, []).append(member)

            # Try to open as image
            img = read_image_from_zip(zf, member)
            if img is None:
                corrupted.append({"member": member, "reason": "cannot decode image"})
                continue

            # Check dimensions
            w, h_px = img.size
            if w < min_dim or h_px < min_dim:
                corrupted.append(
                    {
                        "member": member,
                        "reason": f"too small: {w}x{h_px} < {min_dim}",
                    }
                )
                continue

            # Check uniformity
            arr = np.array(img.convert("L"), dtype=float)
            if _is_uniform(arr, std_thresh):
                uniform.append(member)
                corrupted.append({"member": member, "reason": "uniform (std < threshold)"})
                continue

            valid_members.add(member)

    # Find duplicates (groups with > 1 member)
    duplicates = [members for members in hash_map.values() if len(members) > 1]

    logger.info(
        "Integrity done — valid=%d, corrupted=%d, uniform=%d, duplicate_groups=%d",
        len(valid_members),
        len(corrupted),
        len(uniform),
        len(duplicates),
    )

    return {
        "corrupted_or_invalid": corrupted,
        "corrupted_count": len(corrupted),
        "corrupted_sample": corrupted[:20],
        "uniform_images": uniform,
        "duplicate_files": duplicates,
        "duplicate_count": len(duplicates),
        "valid_members": valid_members,
    }



In [ ]:
%%writefile dq_agent/modality.py
"""Module 3 — Modality classification: complete / mammo_only / us_only / empty."""

from __future__ import annotations

import logging
from typing import Any

from .config import Config

logger = logging.getLogger(__name__)

# Canonical status values
STATUS_COMPLETE = "complete"
STATUS_MAMMO_ONLY = "mammo_only"
STATUS_US_ONLY = "us_only"
STATUS_EMPTY = "empty"
STATUS_PARTIAL = "partial"  # has both modalities but missing report


def classify_patients(
    cfg: Config,
    patients: dict[str, dict],
    valid_members: set[str],
) -> dict[str, dict]:
    """
    Classify each patient by modality completeness.

    Parameters
    ----------
    cfg : Config
    patients : zip inventory from scan_zip()
    valid_members : set of zip members that passed integrity checks

    Returns
    -------
    dict mapping patient_id -> {
        "status": str,
        "n_valid_us": int,
        "n_valid_mammo": int,
        "has_report": bool,
        "valid_us_members": list,
        "valid_mammo_members": list,
    }
    """
    result: dict[str, dict] = {}
    require_report = cfg.modality_policy.require_report

    for pid, inv in patients.items():
        valid_us = [m for m in inv["us"] if m in valid_members]
        valid_mammo = [m for m in inv["mammo"] if m in valid_members]
        has_report = len(inv["report"]) > 0

        has_us = len(valid_us) > 0
        has_mammo = len(valid_mammo) > 0

        # Status based ONLY on images — report presence is metadata, not a gate
        if has_us and has_mammo:
            status = STATUS_COMPLETE
        elif has_mammo:
            status = STATUS_MAMMO_ONLY
        elif has_us:
            status = STATUS_US_ONLY
        else:
            status = STATUS_EMPTY

        result[pid] = {
            "status": status,
            "n_valid_us": len(valid_us),
            "n_valid_mammo": len(valid_mammo),
            "has_report": has_report,
            "valid_us_members": valid_us,
            "valid_mammo_members": valid_mammo,
        }

    counts = {}
    for v in result.values():
        s = v["status"]
        counts[s] = counts.get(s, 0) + 1

    logger.info("Modality classification — %s", counts)
    return result


def modality_summary(modalities: dict[str, dict]) -> dict[str, int]:
    """Return status counts."""
    counts: dict[str, int] = {}
    for v in modalities.values():
        s = v["status"]
        counts[s] = counts.get(s, 0) + 1
    return counts


def filter_for_training(
    modalities: dict[str, dict],
) -> tuple[list[str], list[str]]:
    """
    Split patients into multimodal (for training) and partial (for unimodal use).

    Returns
    -------
    (multimodal_ids, partial_ids)
    """
    multimodal = [pid for pid, v in modalities.items() if v["status"] == STATUS_COMPLETE]
    partial = [pid for pid, v in modalities.items() if v["status"] != STATUS_COMPLETE]
    logger.info(
        "Training split — multimodal=%d, partial/excluded=%d",
        len(multimodal),
        len(partial),
    )
    return multimodal, partial



In [ ]:
%%writefile dq_agent/plots.py
"""Profiling plots: BI-RADS distribution, images per patient, modality breakdown, before/after."""

from __future__ import annotations

import logging
import zipfile
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import numpy as np

from .config import Config
from .zip_io import read_image_from_zip

logger = logging.getLogger(__name__)


def _save(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    logger.info("Plot saved: %s", path)


def plot_birads_distribution(birads: dict[str, int], out_dir: Path) -> None:
    """Bar chart of BI-RADS class distribution."""
    fig, ax = plt.subplots(figsize=(8, 5))
    classes = list(birads.keys())
    counts = list(birads.values())
    colors = plt.cm.tab10(range(len(classes)))
    ax.bar(classes, counts, color=colors)
    ax.set_title("Distribution des classes BI-RADS")
    ax.set_xlabel("Classe BI-RADS")
    ax.set_ylabel("Nombre de patients")
    for i, v in enumerate(counts):
        ax.text(i, v + 0.3, str(v), ha="center", fontsize=9)
    ax.set_ylim(0, max(counts) * 1.15 if counts else 1)
    _save(fig, out_dir / "birads_distribution.png")


def plot_images_per_patient(profile: dict[str, Any], out_dir: Path) -> None:
    """Histograms of US and mammo image counts per patient."""
    per_patient = profile.get("per_patient", [])
    if not per_patient:
        return

    us_counts = [p["n_us"] for p in per_patient]
    mammo_counts = [p["n_mammo"] for p in per_patient]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, data, title in [
        (axes[0], us_counts, "Images échographie (US) par patient"),
        (axes[1], mammo_counts, "Images mammographie par patient"),
    ]:
        bins = range(0, max(data) + 2) if data else [0, 1]
        ax.hist(data, bins=bins, color="#4C72B0", edgecolor="white", rwidth=0.85)
        ax.set_title(title)
        ax.set_xlabel("Nombre d'images")
        ax.set_ylabel("Nombre de patients")
        ax.set_xticks(list(bins))

    fig.suptitle("Nombre d'images par patient", fontsize=13)
    _save(fig, out_dir / "images_per_patient.png")


def plot_modality_distribution(modality_counts: dict[str, int], out_dir: Path) -> None:
    """Pie chart of modality status."""
    labels = list(modality_counts.keys())
    sizes = list(modality_counts.values())
    if not any(sizes):
        return

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(
        sizes,
        labels=labels,
        autopct="%1.1f%%",
        startangle=140,
        colors=plt.cm.Set2(range(len(labels))),
    )
    ax.set_title("Répartition des statuts de modalité")
    _save(fig, out_dir / "modality_distribution.png")


def plot_image_size_histogram(profile: dict[str, Any], zip_path: Path, out_dir: Path) -> None:
    """Sample image dimensions histogram (from first 50 patients)."""
    per_patient = profile.get("per_patient", [])
    widths, heights = [], []

    sample_members: list[str] = []
    for pp in per_patient[:50]:
        pid = pp["patient_id"]
        # We can't easily get members back from profile alone; skip silently
        pass

    # Quick scan from zip
    try:
        from .zip_io import scan_zip
        with zipfile.ZipFile(zip_path, "r") as zf:
            names = zf.namelist()
            sample = [n for n in names if n.lower().endswith(".png")][:200]
            for member in sample:
                try:
                    from PIL import Image
                    import io
                    data = zf.read(member)
                    img = Image.open(io.BytesIO(data))
                    widths.append(img.width)
                    heights.append(img.height)
                except Exception:
                    pass
    except Exception as exc:
        logger.warning("Could not sample image sizes: %s", exc)
        return

    if not widths:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(widths, bins=30, color="#DD8452", edgecolor="white")
    axes[0].set_title("Distribution largeur (pixels)")
    axes[0].set_xlabel("Largeur")
    axes[0].set_ylabel("Nombre d'images")

    axes[1].hist(heights, bins=30, color="#55A868", edgecolor="white")
    axes[1].set_title("Distribution hauteur (pixels)")
    axes[1].set_xlabel("Hauteur")

    fig.suptitle("Tailles des images originales (échantillon 200)", fontsize=13)
    _save(fig, out_dir / "image_size_histogram.png")


def plot_before_after(
    cfg: Config,
    patients: dict[str, dict],
    modalities: dict[str, dict],
    out_dir: Path,
) -> None:
    """
    Show before/after preprocessing for 1 mammo + 1 US patient.
    Reads directly from ZIP without saving full preprocessed image.
    """
    from .preprocess import _to_gray_array, preprocess_mammo, preprocess_us

    zip_path = cfg.zip_path

    # Pick first complete patients
    mammo_pid = us_pid = None
    for pid, info in modalities.items():
        if info["valid_mammo_members"] and mammo_pid is None:
            mammo_pid = pid
        if info["valid_us_members"] and us_pid is None:
            us_pid = pid
        if mammo_pid and us_pid:
            break

    with zipfile.ZipFile(zip_path, "r") as zf:
        for pid, member_key, title, preprocess_fn in [
            (mammo_pid, "valid_mammo_members", "Mammographie", preprocess_mammo),
            (us_pid, "valid_us_members", "Échographie", preprocess_us),
        ]:
            if pid is None:
                continue
            members = modalities[pid][member_key]
            if not members:
                continue

            img = read_image_from_zip(zf, members[0])
            if img is None:
                continue

            arr_before = _to_gray_array(img)
            arr_after = preprocess_fn(arr_before.copy(), cfg)

            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
            axes[0].imshow(arr_before, cmap="gray")
            axes[0].set_title(f"{title} — Original")
            axes[0].axis("off")

            axes[1].imshow(arr_after, cmap="gray")
            axes[1].set_title(f"{title} — Après prétraitement (512×512)")
            axes[1].axis("off")

            fig.suptitle(f"Avant / Après — {title}", fontsize=13)
            fname = "before_after_mammo.png" if "mammo" in member_key else "before_after_us.png"
            _save(fig, out_dir / fname)


def generate_all_plots(
    cfg: Config,
    profile: dict[str, Any],
    modality_counts: dict[str, int],
    patients: dict[str, dict],
    modalities: dict[str, dict],
) -> None:
    """Generate and save all profiling plots."""
    out_dir = cfg.plots_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    plot_birads_distribution(profile["birads_distribution"], out_dir)
    plot_images_per_patient(profile, out_dir)
    plot_modality_distribution(modality_counts, out_dir)
    plot_image_size_histogram(profile, cfg.zip_path, out_dir)
    plot_before_after(cfg, patients, modalities, out_dir)
    logger.info("All plots saved to %s", out_dir)



In [ ]:
%%writefile dq_agent/preprocess.py
"""Module 4 — Preprocessing: normalization, denoising, CLAHE, resize."""

from __future__ import annotations

import logging
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image
from tqdm import tqdm

from .config import Config
from .zip_io import read_image_from_zip

logger = logging.getLogger(__name__)


def _to_gray_array(img: Image.Image) -> np.ndarray:
    """Convert PIL image to grayscale float32 array [0, 1]."""
    gray = img.convert("L")
    arr = np.array(gray, dtype=np.float32) / 255.0
    return arr


def _normalize_01(arr: np.ndarray) -> np.ndarray:
    """Normalize array to [0, 1]."""
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-8:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)


def _median_denoise(arr: np.ndarray, ksize: int) -> np.ndarray:
    """Apply median filter (for ultrasound speckle noise)."""
    try:
        import cv2  # type: ignore
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        filtered = cv2.medianBlur(u8, ksize)
        return filtered.astype(np.float32) / 255.0
    except ImportError:
        from scipy.ndimage import median_filter  # type: ignore
        return median_filter(arr, size=ksize)


def _nlm_denoise(arr: np.ndarray, h: int) -> np.ndarray:
    """Apply Non-Local Means denoising (for mammography)."""
    try:
        import cv2  # type: ignore
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        denoised = cv2.fastNlMeansDenoising(u8, h=h, templateWindowSize=7, searchWindowSize=21)
        return denoised.astype(np.float32) / 255.0
    except ImportError:
        logger.warning("cv2 not available, skipping NLM denoising")
        return arr


def _apply_clahe(arr: np.ndarray, clip_limit: float, tile_grid: int) -> np.ndarray:
    """Apply CLAHE to enhance local contrast (for mammography)."""
    try:
        import cv2  # type: ignore
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid))
        enhanced = clahe.apply(u8)
        return enhanced.astype(np.float32) / 255.0
    except ImportError:
        logger.warning("cv2 not available, skipping CLAHE")
        return arr


def _resize(arr: np.ndarray, target: int) -> np.ndarray:
    """Resize to target×target using INTER_AREA (best for downscaling)."""
    try:
        import cv2  # type: ignore
        u8 = (arr * 255).clip(0, 255).astype(np.uint8)
        resized = cv2.resize(u8, (target, target), interpolation=cv2.INTER_AREA)
        return resized
    except ImportError:
        pil = Image.fromarray((arr * 255).astype(np.uint8))
        pil = pil.resize((target, target), Image.LANCZOS)
        return np.array(pil)


def preprocess_us(arr: np.ndarray, cfg: Config) -> np.ndarray:
    """Full preprocessing pipeline for ultrasound images."""
    arr = _normalize_01(arr)
    arr = _median_denoise(arr, cfg.preprocessing.us_median_ksize)
    arr = _normalize_01(arr)
    return _resize(arr, cfg.preprocessing.target_size)


def preprocess_mammo(arr: np.ndarray, cfg: Config) -> np.ndarray:
    """Full preprocessing pipeline for mammography images."""
    arr = _normalize_01(arr)
    arr = _nlm_denoise(arr, cfg.preprocessing.nlm_h)
    arr = _apply_clahe(
        arr,
        cfg.preprocessing.mammo_clahe_clip_limit,
        cfg.preprocessing.mammo_clahe_tile_grid,
    )
    arr = _normalize_01(arr)
    return _resize(arr, cfg.preprocessing.target_size)


def process_patient(
    pid: str,
    pseudo_id: str,
    modality_info: dict,
    cfg: Config,
    zf: zipfile.ZipFile,
) -> dict[str, Any]:
    """
    Preprocess all images for one patient and save to validated/.

    Returns metadata dict (image paths, counts).
    Skips already-processed images (idempotent).
    """
    mammo_dir = cfg.mammo_dir
    us_dir = cfg.us_dir
    mammo_dir.mkdir(parents=True, exist_ok=True)
    us_dir.mkdir(parents=True, exist_ok=True)

    saved_us: list[str] = []
    saved_mammo: list[str] = []

    # --- Ultrasound ---
    for idx, member in enumerate(modality_info["valid_us_members"]):
        out_name = f"{pseudo_id}_us_{idx}.png"
        out_path = us_dir / out_name

        if out_path.exists():
            saved_us.append(str(out_path.relative_to(cfg.validated_dir)))
            continue

        img = read_image_from_zip(zf, member)
        if img is None:
            logger.warning("Skip US image (unreadable): %s", member)
            continue

        arr = _to_gray_array(img)
        processed = preprocess_us(arr, cfg)
        Image.fromarray(processed, mode="L").save(out_path, format="PNG")
        saved_us.append(str(out_path.relative_to(cfg.validated_dir)))

    # --- Mammography ---
    for idx, member in enumerate(modality_info["valid_mammo_members"]):
        out_name = f"{pseudo_id}_mammo_{idx}.png"
        out_path = mammo_dir / out_name

        if out_path.exists():
            saved_mammo.append(str(out_path.relative_to(cfg.validated_dir)))
            continue

        img = read_image_from_zip(zf, member)
        if img is None:
            logger.warning("Skip mammo image (unreadable): %s", member)
            continue

        arr = _to_gray_array(img)
        processed = preprocess_mammo(arr, cfg)
        Image.fromarray(processed, mode="L").save(out_path, format="PNG")
        saved_mammo.append(str(out_path.relative_to(cfg.validated_dir)))

    return {
        "pseudo_id": pseudo_id,
        "us_paths": saved_us,
        "mammo_paths": saved_mammo,
        "n_us": len(saved_us),
        "n_mammo": len(saved_mammo),
    }



In [ ]:
%%writefile dq_agent/profiling.py
"""Module 1 — Profiling: dataset statistics and BI-RADS distribution."""

from __future__ import annotations

import json
import logging
import re
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm import tqdm

from .config import Config
from .zip_io import read_image_from_zip, scan_zip

logger = logging.getLogger(__name__)

# Pattern: {digits}y{suffix} — suffix encodes study year (e.g. "19" = 2019)
_ID_PATTERN = re.compile(r"^(\d+)y(\d{2,3})$")


def _parse_age_from_id(pid: str) -> int | None:
    """Extract year suffix from patient ID (not the real age — just the year code)."""
    m = _ID_PATTERN.match(pid)
    if m:
        return int(m.group(2))
    return None


def profile_dataset(
    cfg: Config,
    patients: dict[str, dict],
    labels: dict[str, str],
    n_total_zip_patients: int | None = None,
) -> dict[str, Any]:
    """
    Compute dataset-level profiling statistics.

    Parameters
    ----------
    cfg : Config
    patients : output of zip_io.scan_zip() (may be limited)
    labels : dict mapping patient_id -> BI-RADS label (from validation step)
    n_total_zip_patients : total patient dirs in ZIP (before --limit). If None, uses len(patients).

    Returns
    -------
    dict with profiling stats
    """
    zip_path = cfg.zip_path
    n_zip = n_total_zip_patients if n_total_zip_patients is not None else len(patients)
    n_tsv = len(set(labels.values()) - {"UNKNOWN"})  # patients actually in TSV
    # n_patients_in_annotations is set by validation and passed through labels
    # But labels dict includes patients with UNKNOWN label (not in TSV)
    # So: patients with real labels = patients where label != UNKNOWN
    n_annotated = sum(1 for v in labels.values() if v != "UNKNOWN")
    stats: dict[str, Any] = {
        "n_patient_folders": n_zip,
        "n_patients_in_annotations": n_annotated,
        "ecart_patients": abs(n_zip - n_annotated),
        "patients_only_in_folders": sorted(set(patients) - set(labels)),
        "patients_only_in_annotations": sorted(set(labels) - set(patients)),
        "image_extensions": {},
        "modality_counts": {"us": 0, "mammo": 0},
        "per_patient": [],
        "birads_distribution": {},
        "year_suffix_distribution": {},
        "images_without_report": 0,
    }

    ext_counter: dict[str, int] = {}
    birads_counter: dict[str, int] = {}
    year_counter: dict[str, int] = {}

    us_counts: list[int] = []
    mammo_counts: list[int] = []

    logger.info("Profiling %d patients …", len(patients))

    with zipfile.ZipFile(zip_path, "r") as zf:
        for pid, inv in tqdm(patients.items(), desc="Profiling", unit="patient"):
            n_us = len(inv["us"])
            n_mammo = len(inv["mammo"])
            has_report = len(inv["report"]) > 0
            label = labels.get(pid, "UNKNOWN")

            us_counts.append(n_us)
            mammo_counts.append(n_mammo)

            if not has_report:
                stats["images_without_report"] += 1

            # Count extensions
            for member in inv["us"] + inv["mammo"]:
                ext = Path(member).suffix.lower().lstrip(".")
                ext_counter[ext] = ext_counter.get(ext, 0) + 1

            # Sample one US + one mammo image for intensity stats
            us_intensity: float | None = None
            mammo_intensity: float | None = None

            if inv["us"]:
                img = read_image_from_zip(zf, inv["us"][0])
                if img is not None:
                    arr = np.array(img.convert("L"), dtype=float)
                    us_intensity = float(arr.mean())

            if inv["mammo"]:
                img = read_image_from_zip(zf, inv["mammo"][0])
                if img is not None:
                    arr = np.array(img.convert("L"), dtype=float)
                    mammo_intensity = float(arr.mean())

            # Year suffix
            yr = _parse_age_from_id(pid)
            if yr is not None:
                year_counter[str(yr)] = year_counter.get(str(yr), 0) + 1

            # BI-RADS
            birads_counter[label] = birads_counter.get(label, 0) + 1

            stats["per_patient"].append(
                {
                    "patient_id": pid,  # will be pseudonymised later
                    "n_us": n_us,
                    "n_mammo": n_mammo,
                    "has_report": has_report,
                    "birads": label,
                    "year_suffix": yr,
                    "us_mean_intensity": us_intensity,
                    "mammo_mean_intensity": mammo_intensity,
                }
            )

    stats["image_extensions"] = ext_counter
    stats["modality_counts"]["us"] = sum(us_counts)
    stats["modality_counts"]["mammo"] = sum(mammo_counts)
    stats["birads_distribution"] = dict(sorted(birads_counter.items()))
    stats["year_suffix_distribution"] = dict(sorted(year_counter.items(), key=lambda x: int(x[0])))

    if us_counts:
        stats["us_per_patient"] = {
            "min": int(min(us_counts)),
            "max": int(max(us_counts)),
            "mean": round(float(np.mean(us_counts)), 2),
            "std": round(float(np.std(us_counts)), 2),
        }
    if mammo_counts:
        stats["mammo_per_patient"] = {
            "min": int(min(mammo_counts)),
            "max": int(max(mammo_counts)),
            "mean": round(float(np.mean(mammo_counts)), 2),
            "std": round(float(np.std(mammo_counts)), 2),
        }

    logger.info(
        "Profiling done — %d patients, BI-RADS classes: %s",
        len(patients),
        list(stats["birads_distribution"].keys()),
    )
    return stats


def save_profile(profile: dict[str, Any], out_dir: Path) -> None:
    """Persist profiling data (without real IDs in the public version)."""
    out_dir.mkdir(parents=True, exist_ok=True)
    # Save public version (no patient IDs)
    public = {k: v for k, v in profile.items() if k != "per_patient"}
    with open(out_dir / "profile_summary.json", "w", encoding="utf-8") as f:
        json.dump(public, f, indent=2, ensure_ascii=False)
    logger.info("Profile summary saved to %s", out_dir / "profile_summary.json")



In [ ]:
%%writefile dq_agent/report.py
"""Module 7 — Report: generate quality_report.json and quality_report.md."""

from __future__ import annotations

import importlib.metadata
import json
import logging
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

logger = logging.getLogger(__name__)

_LIBRARY_VERSIONS = ["Pillow", "numpy", "pandas", "scikit-learn", "tqdm", "pyyaml", "pydicom"]


def _get_versions() -> dict[str, str]:
    versions: dict[str, str] = {}
    for lib in _LIBRARY_VERSIONS:
        try:
            versions[lib] = importlib.metadata.version(lib)
        except Exception:
            versions[lib] = "N/A"
    return versions


def build_report(
    profile: dict[str, Any],
    integrity_result: dict[str, Any],
    modality_counts: dict[str, int],
    validation_result: dict[str, Any],
    preprocessing_params: dict[str, Any],
    n_multimodal_ready: int,
    reports_stats: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Assemble the quality_report.json payload."""
    report: dict[str, Any] = {
        "date_generated": datetime.now(timezone.utc).isoformat(),
        "library_versions": _get_versions(),
        # Dataset counts
        "n_patient_folders": profile["n_patient_folders"],
        "n_patients_in_annotations": validation_result.get("n_patients_in_annotations", 0),
        "ecart_patients": profile["ecart_patients"],
        "patients_only_in_folders": profile["patients_only_in_folders"],
        "patients_only_in_annotations": profile["patients_only_in_annotations"],
        # Modality
        "modality_status": modality_counts,
        "patients_ready_multimodal": n_multimodal_ready,
        # BI-RADS
        "birads_distribution": profile["birads_distribution"],
        # Integrity
        "corrupted_or_invalid_count": integrity_result["corrupted_count"],
        "corrupted_or_invalid_sample": integrity_result["corrupted_sample"],
        "duplicate_files_count": integrity_result["duplicate_count"],
        "duplicate_groups": integrity_result["duplicate_files"],
        # Annotations quality
        "patients_without_label": validation_result["patients_without_label"],
        "labels_without_folder": validation_result["labels_without_folder"],
        "duplicate_ids_in_annotations": validation_result["duplicate_ids_in_annotations"],
        "malformed_labels": validation_result["malformed_labels"],
        "tsv_columns_detected": validation_result.get("tsv_columns", []),
        "id_column_detected": validation_result.get("id_column"),
        "label_column_detected": validation_result.get("label_column"),
        "extra_tsv_columns": validation_result.get("extra_columns", []),
        # Dataset stats
        "image_extensions": profile["image_extensions"],
        "us_per_patient": profile.get("us_per_patient", {}),
        "mammo_per_patient": profile.get("mammo_per_patient", {}),
        "patients_without_report_pdf": profile.get("images_without_report", 0),
        # Reports extraction
        "reports_stats": reports_stats or {},
        "label_leakage_risk_pct": (reports_stats or {}).get("label_leakage_risk_pct", None),
        # Preprocessing
        "parametres_preprocessing": preprocessing_params,
        # Validation issues
        "all_issues": validation_result.get("issues", []),
    }
    return report


def save_report(report: dict[str, Any], out_dir: Path) -> None:
    """Save JSON and Markdown reports."""
    out_dir.mkdir(parents=True, exist_ok=True)

    json_path = out_dir / "quality_report.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    logger.info("JSON report saved: %s", json_path)

    md_path = out_dir / "quality_report.md"
    _write_markdown(report, md_path)
    logger.info("Markdown report saved: %s", md_path)


def _fmt_list(lst: list, max_items: int = 10) -> str:
    if not lst:
        return "_aucun_"
    shown = [str(x) for x in lst[:max_items]]
    rest = len(lst) - max_items
    result = ", ".join(shown)
    if rest > 0:
        result += f" _… et {rest} autres_"
    return result


def _write_markdown(report: dict[str, Any], path: Path) -> None:
    lines: list[str] = []

    def h(level: int, text: str) -> None:
        lines.append("\n" + "#" * level + " " + text + "\n")

    def row(*cells: Any) -> str:
        return "| " + " | ".join(str(c) for c in cells) + " |"

    h(1, "Rapport de Qualité — Data & Quality Agent (BCMID)")
    lines.append(f"**Généré le :** {report['date_generated']}\n")

    h(2, "1. Résumé Dataset")
    lines.append(row("Métrique", "Valeur"))
    lines.append(row("---", "---"))
    lines.append(row("Dossiers patients (ZIP)", report["n_patient_folders"]))
    lines.append(row("Patients dans les annotations (TSV)", report["n_patients_in_annotations"]))
    lines.append(row("**Écart détecté**", f"**{report['ecart_patients']}**"))
    lines.append(row("Patients prêts (multimodal complet)", report["patients_ready_multimodal"]))
    lines.append(row("Patients sans rapport PDF", report["patients_without_report_pdf"]))

    h(2, "2. Distribution BI-RADS")
    lines.append(row("Classe BI-RADS", "Nombre de patients"))
    lines.append(row("---", "---"))
    for k, v in report["birads_distribution"].items():
        lines.append(row(k, v))

    h(2, "3. Statuts de Modalité")
    lines.append(row("Statut", "Nombre"))
    lines.append(row("---", "---"))
    for k, v in report["modality_status"].items():
        lines.append(row(k, v))

    h(2, "4. Intégrité des Images")
    lines.append(row("Indicateur", "Valeur"))
    lines.append(row("---", "---"))
    lines.append(row("Images corrompues / invalides", report["corrupted_or_invalid_count"]))
    lines.append(row("Groupes de doublons exacts", report["duplicate_files_count"]))

    if report["corrupted_or_invalid_sample"]:
        h(3, "Exemples d'images invalides (max 20)")
        lines.append(row("Fichier", "Raison"))
        lines.append(row("---", "---"))
        for item in report["corrupted_or_invalid_sample"]:
            lines.append(row(item.get("member", "?"), item.get("reason", "?")))

    h(2, "5. Annotations BI-RADS")
    lines.append(row("Indicateur", "Valeur"))
    lines.append(row("---", "---"))
    lines.append(row("Colonnes TSV détectées", ", ".join(report["tsv_columns_detected"])))
    lines.append(row("Colonne ID détectée", report["id_column_detected"] or "N/A"))
    lines.append(row("Colonne label détectée", report["label_column_detected"] or "N/A"))
    lines.append(row("Patients sans label", len(report["patients_without_label"])))
    lines.append(row("Labels sans dossier", len(report["labels_without_folder"])))
    lines.append(row("IDs dupliqués dans TSV", len(report["duplicate_ids_in_annotations"])))
    lines.append(row("Labels mal formés", len(report["malformed_labels"])))

    if report["patients_without_label"]:
        lines.append(f"\n**Patients sans label :** {_fmt_list(report['patients_without_label'])}\n")
    if report["labels_without_folder"]:
        lines.append(f"\n**Labels sans dossier :** {_fmt_list(report['labels_without_folder'])}\n")

    h(2, "6. Statistiques Images")
    for modality, key in [("Échographie (US)", "us_per_patient"), ("Mammographie", "mammo_per_patient")]:
        stats = report.get(key, {})
        if stats:
            lines.append(f"\n**{modality}** — min={stats.get('min')}, max={stats.get('max')}, "
                         f"moy={stats.get('mean')}, écart-type={stats.get('std')}\n")

    h(2, "7. Paramètres de Prétraitement")
    lines.append(row("Paramètre", "Valeur"))
    lines.append(row("---", "---"))
    for k, v in report["parametres_preprocessing"].items():
        lines.append(row(k, v))

    h(2, "8. Versions des Librairies")
    lines.append(row("Librairie", "Version"))
    lines.append(row("---", "---"))
    for lib, ver in report["library_versions"].items():
        lines.append(row(lib, ver))

    path.write_text("\n".join(lines), encoding="utf-8")



In [ ]:
%%writefile dq_agent/reports.py
"""
Module reports — PDF text extraction, PII removal, label leakage detection.

For each patient:
  1. Extract text from PDF report via pdfplumber.
  2. Strip identifying information (names, dates, numeric IDs, real patient ID).
  3. Save as reports/<pseudo_id>.txt in validated/.
  4. Detect whether the text mentions BI-RADS (label leakage risk for Vision agent).
"""

from __future__ import annotations

import io
import logging
import re
import zipfile
from pathlib import Path
from typing import Any

logger = logging.getLogger(__name__)

# ── PII removal patterns ──────────────────────────────────────────────────────
_PII_PATTERNS: list[tuple[str, str]] = [
    # Dates: DD/MM/YYYY, DD-MM-YYYY, MM/DD/YYYY, YYYY-MM-DD, "le 12 janvier 2021"
    (r"\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b", "<DATE>"),
    (r"\b\d{4}[/\-\.]\d{1,2}[/\-\.]\d{1,2}\b", "<DATE>"),
    (r"\b\d{1,2}\s+(janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)\s+\d{4}\b",
     "<DATE>", ),
    # Phone numbers
    (r"\b(?:\+?\d[\s\-]?){8,15}\b", "<PHONE>"),
    # Emails
    (r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b", "<EMAIL>"),
    # Social security / ID numbers (long digit sequences 6-20 chars)
    (r"\b\d{6,20}\b", "<ID_NUM>"),
    # Names heuristic: two or more consecutive Capitalized words (Title Case)
    (r"\b([A-ZÁÉÈÊÀÙÛÔÎÂ][a-záéèêàùûôîâ]+)(\s+[A-ZÁÉÈÊÀÙÛÔÎÂ][a-záéèêàùûôîâ]+){1,3}\b",
     "<NOM>"),
]

# ── BI-RADS mention detection ─────────────────────────────────────────────────
_BIRADS_MENTION = re.compile(
    r"bi[\s\-]*rads|birads|acr\s*\d|cat[eé]gorie\s*\d|classification\s*\d",
    re.IGNORECASE,
)


def _strip_pii(text: str, real_id: str) -> str:
    """Remove PII from extracted text. Also removes the real patient ID."""
    # Remove real patient ID first (most specific)
    text = re.sub(re.escape(real_id), "<PATIENT_ID>", text, flags=re.IGNORECASE)
    # Remove year suffix pattern (NNNNyNN)
    text = re.sub(r"\b\d+y\d+\b", "<PATIENT_ID>", text)

    for pattern_args in _PII_PATTERNS:
        if len(pattern_args) == 3:
            pattern, replacement, *_ = pattern_args
        else:
            pattern, replacement = pattern_args
        try:
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE | re.UNICODE)
        except re.error:
            pass
    return text


def extract_pdf_text(pdf_bytes: bytes) -> tuple[str, str]:
    """
    Extract text from PDF bytes using pdfplumber.

    Returns
    -------
    (text, status)  where status is 'ok' | 'scanned' | 'empty' | 'error'
    """
    try:
        import pdfplumber  # type: ignore
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            pages_text = []
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    pages_text.append(t)
            full_text = "\n".join(pages_text).strip()
    except Exception as exc:
        logger.warning("pdfplumber error: %s", exc)
        return "", "error"

    if not full_text:
        return "", "scanned"   # PDF with no extractable text (image-based)
    return full_text, "ok"


def has_birads_mention(text: str) -> bool:
    """Return True if text contains a BI-RADS classification mention."""
    return bool(_BIRADS_MENTION.search(text))


def process_reports(
    patients: dict[str, dict],
    mapping: dict[str, str],
    zip_path: Path,
    reports_dir: Path,
) -> dict[str, Any]:
    """
    Extract, anonymize, and save all patient PDF reports.

    Parameters
    ----------
    patients : zip inventory from scan_zip()
    mapping  : real_id -> pseudo_id
    zip_path : path to the BCMID ZIP
    reports_dir : output directory for .txt files

    Returns
    -------
    dict with summary stats for quality_report
    """
    from tqdm import tqdm

    reports_dir.mkdir(parents=True, exist_ok=True)

    n_ok = 0
    n_scanned = 0
    n_error = 0
    n_no_pdf = 0
    n_birads_mention = 0
    n_total = 0

    with zipfile.ZipFile(zip_path, "r") as zf:
        for pid, inv in tqdm(patients.items(), desc="Reports", unit="patient"):
            pseudo = mapping.get(pid, pid)
            out_path = reports_dir / f"{pseudo}.txt"

            if not inv["report"]:
                n_no_pdf += 1
                # Write a placeholder
                out_path.write_text("[NO_REPORT_AVAILABLE]\n", encoding="utf-8")
                continue

            # Idempotent
            if out_path.exists():
                text = out_path.read_text(encoding="utf-8")
                if has_birads_mention(text):
                    n_birads_mention += 1
                n_total += 1
                n_ok += 1
                continue

            member = inv["report"][0]
            try:
                pdf_bytes = zf.read(member)
            except Exception as exc:
                logger.warning("Cannot read PDF %s: %s", member, exc)
                n_error += 1
                out_path.write_text("[PDF_READ_ERROR]\n", encoding="utf-8")
                continue

            text, status = extract_pdf_text(pdf_bytes)
            n_total += 1

            if status == "ok":
                n_ok += 1
                # Strip PII
                clean_text = _strip_pii(text, real_id=pid)
                if has_birads_mention(clean_text):
                    n_birads_mention += 1
                out_path.write_text(clean_text, encoding="utf-8")
            elif status == "scanned":
                n_scanned += 1
                out_path.write_text("[SCANNED_PDF_NO_TEXT]\n", encoding="utf-8")
            else:
                n_error += 1
                out_path.write_text("[PDF_ERROR]\n", encoding="utf-8")

    leakage_pct = round(n_birads_mention / n_total * 100, 1) if n_total > 0 else 0.0

    logger.info(
        "Reports — ok=%d, scanned=%d, error=%d, no_pdf=%d | BI-RADS mention: %d/%d (%.1f%%)",
        n_ok, n_scanned, n_error, n_no_pdf, n_birads_mention, n_total, leakage_pct,
    )

    return {
        "reports_extracted_ok": n_ok,
        "reports_scanned_unreadable": n_scanned,
        "reports_error": n_error,
        "reports_no_pdf": n_no_pdf,
        "reports_total_with_pdf": n_total,
        "birads_mention_in_reports": n_birads_mention,
        "label_leakage_risk_pct": leakage_pct,
    }



In [ ]:
%%writefile dq_agent/splits.py
"""Module 8 — Splits: stratified train/val/test split by patient (no data leakage).

label_group is added as metadata but splits are stratified on it (not on raw birads).
The grouping must be validated by the user before calling make_splits.
"""

from __future__ import annotations

import json
import logging
import math
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

from .config import Config

logger = logging.getLogger(__name__)

# ── Proposed clinical grouping (to be validated by user) ─────────────────────
DEFAULT_LABEL_GROUPS: dict[str, str] = {
    "0": "incomplete",
    "1": "benign",
    "2": "benign",
    "3": "probably_benign",
    "4": "suspicious",
    "4A": "suspicious",
    "4B": "suspicious",
    "4C": "suspicious",
    "5": "highly_suspicious",
    "6": "known_malignant",
    "UNKNOWN": "unknown",
    "RARE": "rare",
}


def get_label_group(birads: str, groups: dict[str, str] | None = None) -> str:
    """Map a normalized BI-RADS label to its clinical group."""
    g = groups or DEFAULT_LABEL_GROUPS
    return g.get(birads, "unknown")


def show_distribution(patient_labels: dict[str, str]) -> None:
    """Log BI-RADS distribution and proposed groups."""
    counts = Counter(patient_labels.values())
    logger.info("BI-RADS distribution (raw):")
    for cls in sorted(counts):
        grp = get_label_group(cls)
        logger.info("  BI-RADS %s → group '%s' : %d patients", cls, grp, counts[cls])


def _group_rare_classes(
    patient_labels: dict[str, str],
    threshold: int,
    label_groups: dict[str, str],
) -> tuple[dict[str, str], dict[str, str]]:
    """
    Group BI-RADS classes with fewer than threshold patients into 'RARE'.
    Classes 0 and 1 are NEVER merged with 2 automatically.

    Returns
    -------
    (strat_labels, rare_mapping)
    strat_labels: patient_id -> label_group (used for stratification)
    rare_mapping: original_class -> 'rare' (documented)
    """
    group_counts = Counter(
        get_label_group(lbl, label_groups) for lbl in patient_labels.values()
    )
    rare_groups = {grp for grp, cnt in group_counts.items() if cnt < threshold}

    if rare_groups:
        logger.info(
            "Groups with < %d patients (rare): %s — will be pooled as 'rare' for splits only",
            threshold, sorted(rare_groups),
        )

    strat_labels: dict[str, str] = {}
    rare_mapping: dict[str, str] = {}

    for pid, birads in patient_labels.items():
        grp = get_label_group(birads, label_groups)
        if grp in rare_groups:
            strat_labels[pid] = "rare"
            rare_mapping[birads] = "rare"
        else:
            strat_labels[pid] = grp

    return strat_labels, rare_mapping


def make_splits(
    cfg: Config,
    multimodal_ids: list[str],
    labels: dict[str, str],
    label_groups: dict[str, str] | None = None,
) -> dict[str, Any]:
    """
    Create stratified train/val/test splits by patient.

    Stratification is by label_group. BI-RADS 0 and 1 are NEVER auto-merged with 2.
    No patient appears in more than one split.

    Returns
    -------
    dict with train/val/test lists of patient IDs and metadata.
    """
    rng = random.Random(cfg.splits.random_seed)
    threshold = cfg.splits.rare_class_threshold
    lg = label_groups or DEFAULT_LABEL_GROUPS

    # Separate known vs unknown labels
    patient_labels = {
        pid: labels.get(pid, "UNKNOWN")
        for pid in multimodal_ids
        if labels.get(pid, "UNKNOWN") != "UNKNOWN"
    }
    unknown_ids = [pid for pid in multimodal_ids if labels.get(pid, "UNKNOWN") == "UNKNOWN"]
    if unknown_ids:
        logger.warning(
            "%d patients with UNKNOWN label — added to train set.", len(unknown_ids)
        )

    show_distribution(patient_labels)

    # Group by clinical category for stratification
    strat_labels, rare_mapping = _group_rare_classes(patient_labels, threshold, lg)

    # Stratify buckets
    class_buckets: dict[str, list[str]] = defaultdict(list)
    for pid, grp in strat_labels.items():
        class_buckets[grp].append(pid)

    for grp in class_buckets:
        rng.shuffle(class_buckets[grp])

    train_ids: list[str] = []
    val_ids: list[str] = []
    test_ids: list[str] = []

    train_r = cfg.splits.train_ratio
    val_r = cfg.splits.val_ratio

    for grp, pids in class_buckets.items():
        n = len(pids)
        n_train = max(1, math.floor(n * train_r))
        n_val = max(0, math.floor(n * val_r))
        n_val = min(n_val, n - n_train)
        n_test = n - n_train - n_val

        train_ids.extend(pids[:n_train])
        val_ids.extend(pids[n_train: n_train + n_val])
        test_ids.extend(pids[n_train + n_val:])

    train_ids.extend(unknown_ids)

    rng.shuffle(train_ids)
    rng.shuffle(val_ids)
    rng.shuffle(test_ids)

    # ── No-leakage invariant ──
    assert set(train_ids).isdisjoint(set(val_ids)),  "LEAK: patient(s) in train AND val"
    assert set(train_ids).isdisjoint(set(test_ids)), "LEAK: patient(s) in train AND test"
    assert set(val_ids).isdisjoint(set(test_ids)),   "LEAK: patient(s) in val AND test"

    logger.info(
        "Splits — train=%d, val=%d, test=%d (seed=%d)",
        len(train_ids), len(val_ids), len(test_ids), cfg.splits.random_seed,
    )

    return {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids,
        "n_train": len(train_ids),
        "n_val": len(val_ids),
        "n_test": len(test_ids),
        "random_seed": cfg.splits.random_seed,
        "rare_groups_pooled": rare_mapping,
        "rare_threshold": threshold,
        "unknown_label_patients_in_train": unknown_ids,
        "label_groups_used": lg,
        "class_distribution": {
            "train": dict(Counter(strat_labels.get(p, "unknown") for p in train_ids)),
            "val":   dict(Counter(strat_labels.get(p, "unknown") for p in val_ids)),
            "test":  dict(Counter(strat_labels.get(p, "unknown") for p in test_ids)),
        },
    }


def save_splits(splits: dict[str, Any], out_dir: Path) -> None:
    """Save splits.json."""
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "splits.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(splits, f, indent=2, ensure_ascii=False)
    logger.info("Splits saved: %s", path)



In [ ]:
%%writefile dq_agent/validation.py
"""Module 6 — Validation: join images with BI-RADS labels, detect anomalies."""

from __future__ import annotations

import logging
import re
from pathlib import Path
from typing import Any

import pandas as pd

from .config import Config

logger = logging.getLogger(__name__)

# BI-RADS normalization patterns
_BIRADS_PATTERNS = [
    # Match "BI-RADS 4A", "BIRADS4a", "bi rads 4A" etc.
    re.compile(r"bi[\s\-_]*rads[\s\-_]*(\d[A-Ca-c]?)", re.IGNORECASE),
    # Match bare numbers/letters like "4A", "4a", "4", "0"
    re.compile(r"^(\d[A-Ca-c]?)$", re.IGNORECASE),
]

_VALID_BIRADS = {
    "0", "1", "2", "3",
    "4", "4A", "4B", "4C",
    "5", "6",
}


def normalize_birads(raw: str) -> str:
    """
    Normalize a raw BI-RADS label string to canonical form.

    Examples
    --------
    "4A" -> "4A"
    "4a" -> "4A"
    "BI-RADS 4A" -> "4A"
    "bi rads 4b" -> "4B"
    "4" -> "4"
    """
    raw = str(raw).strip()
    for pat in _BIRADS_PATTERNS:
        m = pat.search(raw)
        if m:
            val = m.group(1).upper()
            return val
    return raw.upper().strip()


def load_annotations(tsv_path: Path) -> tuple[pd.DataFrame, list[str]]:
    """
    Load and inspect the annotations TSV.

    Returns
    -------
    (df, column_names)
    df has columns auto-detected; patient_id_col and birads_col identified heuristically.
    """
    if not tsv_path.exists():
        raise FileNotFoundError(f"Annotations TSV not found: {tsv_path}")

    df = pd.read_csv(tsv_path, sep="\t", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    logger.info("TSV loaded — columns: %s, rows: %d", list(df.columns), len(df))
    return df, list(df.columns)


def detect_id_and_label_columns(df: pd.DataFrame) -> tuple[str, str]:
    """
    Heuristically identify the patient ID column and the BI-RADS label column.
    Works with the real BCMID TSV columns: 'Patient ID', 'BI-RADS Label', etc.
    """
    cols = [c.lower() for c in df.columns]
    orig_cols = list(df.columns)

    id_col = None
    label_col = None

    # ID column heuristics — covers "Patient ID", "patient_id", "id", etc.
    for kw in ["patient id", "patient_id", "patientid", "id", "patient", "case", "subject"]:
        for i, c in enumerate(cols):
            if kw in c and "label" not in c:
                id_col = orig_cols[i]
                break
        if id_col:
            break

    # Label column heuristics — covers "BI-RADS Label", "birads", "label", etc.
    for kw in ["bi-rads", "birads", "bi_rads", "bi rads"]:
        for i, c in enumerate(cols):
            if kw in c:
                label_col = orig_cols[i]
                break
        if label_col:
            break
    # Fallback to any column named "label" or "class"
    if label_col is None:
        for kw in ["label", "class", "category", "grade"]:
            for i, c in enumerate(cols):
                if kw in c and i != cols.index(id_col.lower() if id_col else ""):
                    label_col = orig_cols[i]
                    break
            if label_col:
                break

    # Last-resort fallback
    if id_col is None:
        id_col = orig_cols[0]
        logger.warning("Could not detect ID column — using first column: %s", id_col)
    if label_col is None:
        label_col = orig_cols[-1]
        logger.warning("Could not detect label column — using last column: %s", label_col)

    logger.info("Detected columns — ID: '%s', Label: '%s'", id_col, label_col)
    return id_col, label_col


def validate_labels(
    cfg: Config,
    patients: dict[str, dict],
) -> dict[str, Any]:
    """
    Full validation of BI-RADS labels against patient folders.

    Returns
    -------
    dict with:
      - labels: dict patient_id -> normalized_birads
      - issues: list of issue dicts
      - duplicate_ids_in_annotations: list
      - patients_without_label: list
      - labels_without_folder: list
      - malformed_labels: list
    """
    tsv_path = cfg.tsv_path
    issues: list[dict] = []

    # Handle missing TSV gracefully
    if not tsv_path.exists():
        logger.warning("TSV not found at %s — labels will be UNKNOWN", tsv_path)
        labels = {pid: "UNKNOWN" for pid in patients}
        return {
            "labels": labels,
            "issues": [{"type": "missing_tsv", "detail": str(tsv_path)}],
            "duplicate_ids_in_annotations": [],
            "patients_without_label": list(patients.keys()),
            "labels_without_folder": [],
            "malformed_labels": [],
            "tsv_columns": [],
            "id_column": None,
            "label_column": None,
        }

    df, cols = load_annotations(tsv_path)
    id_col, label_col = detect_id_and_label_columns(df)

    # Capture any extra informative columns
    extra_cols = [c for c in cols if c not in (id_col, label_col)]
    extra_data: dict[str, dict] = {}

    # Detect duplicate IDs in TSV
    dup_ids = df[df.duplicated(subset=[id_col], keep=False)][id_col].tolist()
    if dup_ids:
        issues.append({"type": "duplicate_ids_in_annotations", "ids": list(set(dup_ids))})
        logger.warning("Duplicate IDs in annotations: %s", dup_ids[:10])

    # Deduplicate (keep first)
    df = df.drop_duplicates(subset=[id_col], keep="first")

    # Store extra column data per patient
    for _, row in df.iterrows():
        pid = str(row[id_col]).strip()
        extra_data[pid] = {c: str(row[c]).strip() for c in extra_cols}

    # Normalize labels
    raw_labels = df.set_index(id_col)[label_col].to_dict()
    normalized_labels: dict[str, str] = {}
    malformed: list[dict] = []

    for pid_raw, lbl in raw_labels.items():
        pid = str(pid_raw).strip()
        norm = normalize_birads(str(lbl))
        if norm not in _VALID_BIRADS:
            malformed.append({"patient_id": pid, "raw_label": lbl, "normalized": norm})
            issues.append({"type": "malformed_label", "patient_id": pid, "raw": lbl, "normalized": norm})
        normalized_labels[pid] = norm

    # Cross-join
    folder_ids = set(patients.keys())
    annotation_ids = set(normalized_labels.keys())

    patients_without_label = sorted(folder_ids - annotation_ids)
    labels_without_folder = sorted(annotation_ids - folder_ids)

    for pid in patients_without_label:
        issues.append({"type": "patient_without_label", "patient_id": pid})
        normalized_labels[pid] = "UNKNOWN"

    if labels_without_folder:
        issues.append({"type": "labels_without_folder", "ids": labels_without_folder})

    logger.info(
        "Validation — %d labels loaded, %d without label, %d labels without folder, %d malformed",
        len(raw_labels),
        len(patients_without_label),
        len(labels_without_folder),
        len(malformed),
    )

    return {
        "labels": normalized_labels,
        "issues": issues,
        "duplicate_ids_in_annotations": list(set(dup_ids)),
        "patients_without_label": patients_without_label,
        "labels_without_folder": labels_without_folder,
        "malformed_labels": malformed,
        "tsv_columns": cols,
        "id_column": id_col,
        "label_column": label_col,
        "extra_columns": extra_cols if 'extra_cols' in dir() else [],
        "extra_data": extra_data if 'extra_data' in dir() else {},
        "n_patients_in_annotations": len(df) if 'df' in dir() else 0,
    }



In [ ]:
%%writefile dq_agent/zip_io.py
"""ZIP I/O utilities — streaming access to images without full extraction."""

from __future__ import annotations

import io
import logging
import zipfile
from pathlib import Path, PurePosixPath
from typing import Generator, Iterator

from PIL import Image

logger = logging.getLogger(__name__)

# Folder name aliases for modalities
_US_ALIASES = {"ultrasound", "us", "echo", "echography", "ultrason"}
_MAMMO_ALIASES = {"mammogram", "mammography", "mammo", "mg", "mammographie"}


def _is_us_folder(name: str) -> bool:
    return name.lower() in _US_ALIASES


def _is_mammo_folder(name: str) -> bool:
    return name.lower() in _MAMMO_ALIASES


def _is_image(fname: str) -> bool:
    return fname.lower().endswith((".png", ".jpg", ".jpeg", ".dcm", ".bmp", ".tiff", ".tif"))


def _is_pdf(fname: str) -> bool:
    return fname.lower().endswith(".pdf")


def scan_zip(zip_path: Path) -> dict[str, dict]:
    """
    Scan zip structure and return per-patient inventory.

    Returns
    -------
    dict mapping patient_id -> {
        "us": [zip_member_path, ...],
        "mammo": [zip_member_path, ...],
        "report": [zip_member_path, ...],
        "other": [...],
    }
    """
    patients: dict[str, dict] = {}

    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            parts = PurePosixPath(info.filename).parts
            if len(parts) < 1 or not parts[0]:
                continue

            pid = parts[0]
            if pid not in patients:
                patients[pid] = {"us": [], "mammo": [], "report": [], "other": []}

            if len(parts) == 1:
                # bare folder entry
                continue

            fname = parts[-1]
            if not fname:
                continue  # folder entry

            if len(parts) == 3:
                sub = parts[1]
                if _is_us_folder(sub) and _is_image(fname):
                    patients[pid]["us"].append(info.filename)
                elif _is_mammo_folder(sub) and _is_image(fname):
                    patients[pid]["mammo"].append(info.filename)
                else:
                    patients[pid]["other"].append(info.filename)
            elif len(parts) == 2:
                if _is_pdf(fname):
                    patients[pid]["report"].append(info.filename)
                elif _is_image(fname):
                    patients[pid]["other"].append(info.filename)

    return patients


def read_image_from_zip(zf: zipfile.ZipFile, member: str) -> Image.Image | None:
    """
    Read a single image from an open ZipFile into a PIL Image.

    Handles PNG/JPEG/BMP via Pillow and DICOM via pydicom.
    Returns None on error (logged).
    """
    try:
        data = zf.read(member)
    except Exception as exc:
        logger.warning("Cannot read zip member %s: %s", member, exc)
        return None

    fname = member.lower()

    if fname.endswith(".dcm"):
        try:
            import pydicom
            import numpy as np

            ds = pydicom.dcmread(io.BytesIO(data))
            arr = ds.pixel_array.astype(float)
            # Handle MONOCHROME1 (invert)
            if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
                arr = arr.max() - arr
            arr = (arr - arr.min()) / (arr.ptp() + 1e-8) * 255
            return Image.fromarray(arr.astype("uint8"), mode="L")
        except Exception as exc:
            logger.warning("DICOM read error for %s: %s", member, exc)
            return None
    else:
        try:
            img = Image.open(io.BytesIO(data))
            img.load()  # force decode
            return img
        except Exception as exc:
            logger.warning("Image read error for %s: %s", member, exc)
            return None


def iter_patients(
    zip_path: Path,
    patients: dict[str, dict],
    limit: int | None = None,
) -> Generator[tuple[str, dict, zipfile.ZipFile], None, None]:
    """
    Yield (patient_id, inventory, open_zipfile) for each patient.

    Keeps the ZipFile open across patients for efficiency.
    """
    pids = list(patients.keys())
    if limit is not None:
        pids = pids[:limit]

    with zipfile.ZipFile(zip_path, "r") as zf:
        for pid in pids:
            yield pid, patients[pid], zf



In [ ]:
%%writefile dq_agent/__init__.py
"""Data & Quality Agent for BCMID — Multimodal Medical AI Pipeline."""

__version__ = "1.0.0"
__author__ = "Data & Quality Agent"



In [ ]:
%%writefile dq_agent/__main__.py
"""Allows `python -m dq_agent` to invoke the CLI."""

from .cli import main

if __name__ == "__main__":
    main()



### 2. Configuration
**ATTENTION:** Modifie les chemins ci-dessous pour correspondre à ton Dataset Kaggle.


In [ ]:
%%writefile config.yaml
zip_path: "/kaggle/input/bcmid/BCMID.zip"  # <-- MODIFIER ICI selon le dataset Kaggle
tsv_path: "/kaggle/input/bcmid/BCMID_annotations.tsv" # <-- MODIFIER ICI
work_dir: "/kaggle/working/bcmid_work"
private_dir: "/kaggle/working/bcmid_work"

preprocessing:
  target_size: [512, 512]
  mammo_clahe_clip_limit: 2.0
  mammo_clahe_tile_grid: [8, 8]
  us_median_ksize: 5
  nlm_h: 10
  bit_depth: 8

modality_policy: "keep_all"

splits:
  train_ratio: 0.70
  val_ratio: 0.15
  test_ratio: 0.15
  random_seed: 42



In [ ]:
%%writefile requirements.txt
Pillow>=10.0.0
tqdm>=4.66.0
PyYAML>=6.0
matplotlib>=3.7.0
pdfplumber>=0.10.0
python-dotenv>=1.0.0
pytest>=7.0.0
opencv-python-headless>=4.8.0



### 3. Installation des dépendances


In [ ]:
!pip install -r requirements.txt


### 4. Lancement du Pipeline (Agent 2)
Ceci va prendre environ 1 à 2 heures selon les CPU Kaggle.


In [ ]:
import os
# Définir un sel d'anonymisation pour Kaggle
os.environ["DQ_SALT"] = "KaggleSecretSalt_2026"

!python -m dq_agent run --config config.yaml



### 5. Sauvegarde des résultats
Une fois terminé, le dossier `/kaggle/working/bcmid_work/validated` contient ton dataset prêt pour l'entraînement ! Tu pourras le télécharger ou l'utiliser directement dans la suite de ton code.
